In [10]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
import sys
# df_cafe = pd.read_csv(r"C:\Users\iq750\bootcamp_git\Final-Project-2_Team1\data\대용량\상가정보\소상공인시장진흥공단_상가(상권)정보_서울_202412.csv")
df_sales = pd.read_csv(r"C:\Users\iq750\bootcamp_git\Final-Project-2_Team1\data\서울시 상권분석서비스(추정매출-상권)_2024년.csv")
df_pop = pd.read_csv(r"C:\Users\iq750\bootcamp_git\Final-Project-2_Team1\data\유동인구_건물기반분배_0506.csv")
df_review = pd.read_csv(r"C:\Users\iq750\bootcamp_git\Final-Project-2_Team1\data\점수산출\카페_별점리뷰통합.csv")
df_mapping = pd.read_csv(r"C:\Users\iq750\bootcamp_git\Final-Project-2_Team1\data\카페_상권_매핑_데이터_중복제거.csv") 
df_mapping.rename(columns={'TRDAR_CD':'상권_코드'}, inplace=True)
df_pop = df_pop.merge(df_mapping[['상가업소번호','상호명','지점명']], how = 'left', on = '상가업소번호')
print('fisrt mapping lenth: ', len(df_mapping))
# 1. 병합 준비
df_mapping["상호명_지점"] = df_mapping.apply(
    lambda row: f"{row['상호명']} {str(row['지점명']).strip()}" if pd.notna(row["지점명"]) and str(row["지점명"]).strip()
    else row["상호명"],
    axis=1
)

# 2. 전용면적 점수 병합
# '고유번호'로 연결 - X
# df_area = pd.read_csv(r'C:\Users\iq750\bootcamp_git\Final-Project-2_Team1\data\temp_카페_면적_예상.csv')
# df_area = df_area[['상가업소번호','고유번호']]
# df_mapping = df_mapping.merge(df_area, on = '상가업소번호',how = 'left')
# print('고유번호 결합 후 길이: ' ,len(df_mapping))
# print(df_mapping[['상가업소번호','고유번호']].isna().sum())

# 면적 계산 및 공간 조인
gdf_building = gpd.read_file(r"C:\Users\iq750\bootcamp_git\Final-Project-2_Team1\data\대용량\전체건물정보\AL_D010_11_20250404.shp").to_crs(epsg=4326)
gdf_building_utm = gdf_building.to_crs(epsg=5181)
gdf_building['전용면적'] = gdf_building_utm.geometry.area
gdf_building.rename(columns = {'A2':'고유번호'},inplace=True)

# A12로 중복제거
gdf_building = gdf_building.sort_values('A12', ascending=False)
gdf_building = gdf_building.drop_duplicates(subset=['A3','A5'], keep='first')
# 공간 조인
gdf_mapping = gpd.GeoDataFrame(df_mapping, geometry=gpd.points_from_xy(df_mapping["경도"], df_mapping["위도"]), crs="EPSG:4326")
gdf_joined = gpd.sjoin(gdf_mapping, gdf_building[['전용면적','geometry','고유번호']], how = 'left', predicate='within'  )

# 3. 건물 내 카페 수 세기 및 균등분배
gdf_joined['건물내_카페수'] = gdf_joined.groupby("고유번호")['상가업소번호'].transform("count")
gdf_joined['균등분배_전용면적'] = gdf_joined['전용면적'] / gdf_joined['건물내_카페수']
# 4. 면적 점수 계산 및 병합
gdf_joined["균등분배_전용면적"] = gdf_joined.groupby("상권_코드")["균등분배_전용면적"].transform(lambda x: x.fillna(x.mean()))
gdf_joined["면적_점수"] = gdf_joined.groupby("상권_코드")["균등분배_전용면적"].transform(lambda x: x / x.sum())

df_mapping = pd.merge(
    df_mapping,
    gdf_joined[["상가업소번호", "균등분배_전용면적", "면적_점수"]],
    on="상가업소번호",
    how="left"
)
print('면적 점수 추가 후 길이: ', len(df_mapping))
print(df_mapping['상가업소번호'].value_counts())
df_mapping = df_mapping.drop_duplicates(subset = ['상가업소번호'],keep = 'first') # 중복 제거
df_mapping['면적_점수'] = df_mapping['면적_점수'].fillna(df_mapping['면적_점수'].median())

# print(df_mapping['상가업소번호'].value_counts())
# 중복 check 완료

# 5. 인구 점수 병합
df_pop["상호명_지점"] = df_pop.apply(
    lambda row: f"{row['상호명']} {str(row['지점명']).strip()}" if pd.notna(row["지점명"]) and str(row["지점명"]).strip()
    else row["상호명"],
    axis=1
) # 해당 없음
df_pop.rename(columns={"TRDAR_CD": "상권_코드", "est_총_유동인구_수": "인구_점수"}, inplace=True)
print(df_pop['상가업소번호'].value_counts()) #중복은 없음
# 새로운 mapping 기준 찾아야 할 것 같음
df_mapping = pd.merge(df_mapping, df_pop[["상가업소번호", "상권_코드", "인구_점수"]], on=["상가업소번호", "상권_코드"], how="left")
df_mapping["인구_점수_norm"] = df_mapping.groupby("상권_코드")["인구_점수"].transform(
    lambda x: x / x.sum() if x.sum() != 0 else 0
)
df_mapping["인구_점수_norm"] = df_mapping["인구_점수_norm"].fillna(df_mapping["인구_점수_norm"].median())

print('인구점수 머지 후 길이: ', len(df_mapping))

# 6. 리뷰 점수 병합
df_review["kakao_reviewCount"] = df_review["kakao_reviewCount"].astype(str).str.replace(",", "").astype(float)
df_review["naver_reviewCount"] = df_review["naver_reviewCount"].astype(str).str.replace(",", "").astype(float)
df_review["리뷰수"] = df_review["kakao_reviewCount"] + df_review["naver_reviewCount"]
df_review["별점"] = df_review[["kakao_score", "naver_score"]].mean(axis=1)
df_review = df_review.groupby("상가업소번호", as_index=False)[["리뷰수", "별점"]].mean()

df_mapping = pd.merge(df_mapping, df_review, on="상가업소번호", how="left")
df_mapping["리뷰수"] = df_mapping["리뷰수"].fillna(0)
df_mapping["리뷰_점수"] = df_mapping.groupby("상권_코드")["리뷰수"].transform(lambda x: x / x.sum() if x.sum() != 0 else 0)
print('리뷰 점수 후 길이: ', len(df_mapping))

# 7. 최종 점수 계산
w_area = 0.3
w_pop = 0.3
w_review = 0.4

df_mapping["최종_점수"] = (
    df_mapping["면적_점수"].fillna(0) * w_area +
    df_mapping["인구_점수_norm"].fillna(0) * w_pop +
    df_mapping["리뷰_점수"].fillna(0) * w_review
)

# 8. 추정 매출 계산
df_sales = df_sales[df_sales['서비스_업종_코드'] == "CS100010"]
sales_by_area = df_sales.groupby("상권_코드")["당월_매출_금액"].sum()
df_mapping["상권_매출"] = df_mapping["상권_코드"].map(sales_by_area)

sum_scores = df_mapping.groupby("상권_코드")["최종_점수"].transform("sum")
df_mapping["비율"] = df_mapping["최종_점수"] / sum_scores
df_mapping["비율"] = df_mapping["비율"].fillna(0)
df_mapping["카페_추정매출"] = df_mapping["비율"] * df_mapping["상권_매출"]

print('df_mapping 길이: ', len(df_mapping))

# 7. 결과 저장
print(df_mapping.columns.tolist())
cols_to_save = ["상가업소번호",'법정동코드','지번본번지', '지번부번지', "상호명","지점명", "상권_코드", "균등분배_전용면적", "리뷰수", "별점", "카페_추정매출"]
print('매핑데이터 결측치 확인: \n',df_mapping.isna().sum())

df_mapping[cols_to_save].to_csv("카페_추정매출결과_0511.csv", index=False, encoding="utf-8-sig")


fisrt mapping lenth:  18171
면적 점수 추가 후 길이:  18187
상가업소번호
MA010120220809806690    2
MA0101202211A0099165    2
MA0106202301A1337298    2
MA0101202211A0087493    2
MA0101202406A0205434    2
                       ..
MA010120220807372753    1
MA010120220804283432    1
MA010120220811802533    1
MA010120220813194426    1
MA010120220813491137    1
Name: count, Length: 18171, dtype: int64
상가업소번호
MA0101202409A0132112    1
MA010120220800672495    1
MA0101202406A0465702    1
MA0106202410A0428092    1
MA010120220803375113    1
                       ..
MA0106202406A2465517    1
MA010120220803658119    1
MA010120220810433268    1
MA0101202311A0016316    1
MA010120220812360152    1
Name: count, Length: 18171, dtype: int64
인구점수 머지 후 길이:  18171
리뷰 점수 후 길이:  18171
df_mapping 길이:  18171
['상가업소번호', '상호명', '지점명', '상권_코드', 'TRDAR_CD_N', '상권업종소분류코드', '상권업종소분류명', '시도코드', '시도명', '시군구코드', '시군구명', '행정동코드', '행정동명', '법정동코드', '법정동명', '지번코드', '대지구분코드', '대지구분명', '지번본번지', '지번부번지', '지번주소', '도로명코드', '도로명', '건물본번지', '건물

In [3]:
# 1. 추정 매출 합계 (df_cafe_unique 기준)
estimated_sum = df_cafe_unique.groupby("상권_코드")["카페_추정매출"].sum().reset_index()
estimated_sum.columns = ["상권_코드", "추정매출합계"]

# 2. 실제 상권 매출
actual_sales = df_sales.groupby("상권_코드")["당월_매출_금액"].sum().reset_index()
actual_sales.columns = ["상권_코드", "실제매출"]

# 3. 비교
comparison = pd.merge(estimated_sum, actual_sales, on="상권_코드", how="inner")
comparison["차이"] = comparison["실제매출"] - comparison["추정매출합계"]
comparison["오차비율(%)"] = (comparison["차이"] / comparison["실제매출"]) * 100

# 4. 오차 확인
print("상권별 실제 매출과 추정 매출의 합산 비교:")
print(comparison.sort_values("오차비율(%)", key=abs, ascending=False).head(10))
print(f"오차가 1원 이하인 상권 수: {(comparison['차이'].abs() < 1).sum()} / {len(comparison)}")

상권별 실제 매출과 추정 매출의 합산 비교:
        상권_코드        추정매출합계         실제매출            차이       오차비율(%)
1270  3130050  7.040705e+09   7040704635 -1.907349e-06 -2.709031e-14
335   3110359  7.550200e+09   7550200229  1.907349e-06  2.526223e-14
97    3110101  3.046761e+07     30467608 -7.450581e-09 -2.445410e-14
1346  3130147  3.480941e+10  34809412929 -7.629395e-06 -2.191762e-14
1069  3120074  3.601336e+10  36013355030 -7.629395e-06 -2.118490e-14
519   3110565  4.672376e+09   4672375928 -9.536743e-07 -2.041091e-14
830   3110908  4.728743e+09   4728743330 -9.536743e-07 -2.016761e-14
223   3110240  1.909984e+10  19099842688  3.814697e-06  1.997240e-14
857   3110941  1.211468e+09   1211467559  2.384186e-07  1.968015e-14
380   3110409  1.949965e+10  19499654231 -3.814697e-06 -1.956290e-14
오차가 1원 이하인 상권 수: 1476 / 1476


In [25]:
# 1. 추정매출이 NaN인 샘플 개수
na_sales_cafe = df_cafe_unique[df_cafe_unique["카페_추정매출"].isna()]
print(f"❌ 추정매출이 NaN인 카페 수: {len(na_sales_cafe)}")

# 2. 이들이 속한 상권 코드 확인
missing_area_codes = na_sales_cafe["상권_코드"].unique()
print(f"⚠️ 추정매출 누락 상권 수: {len(missing_area_codes)}")
print("🧾 누락된 상권 코드 목록:")
print(missing_area_codes)

# 3. 실제 추정매출 데이터프레임(df_sales)에 존재하지 않는 상권 코드 찾기
sales_area_codes = df_sales["상권_코드"].unique()
missing_from_sales = [code for code in missing_area_codes if code not in sales_area_codes]
print(f"\n🚫 실제 추정매출 데이터에 존재하지 않는 상권 코드 수: {len(missing_from_sales)}")
print("🗑️ 누락된 상권 코드들:")
print(missing_from_sales)

❌ 추정매출이 NaN인 카페 수: 53
⚠️ 추정매출 누락 상권 수: 34
🧾 누락된 상권 코드 목록:
[3110235 3130034 3110581 3110518 3110841 3110119 3110574 3110384 3110920
 3110705 3110086 3110385 3110682 3110231 3111018 3110761 3130158 3110801
 3110111 3110944 3130155 3130114 3110443 3110914 3110928 3110987 3110629
 3130156 3110321 3110894 3110347 3130187 3110279 3110427]

🚫 실제 추정매출 데이터에 존재하지 않는 상권 코드 수: 34
🗑️ 누락된 상권 코드들:
[np.int64(3110235), np.int64(3130034), np.int64(3110581), np.int64(3110518), np.int64(3110841), np.int64(3110119), np.int64(3110574), np.int64(3110384), np.int64(3110920), np.int64(3110705), np.int64(3110086), np.int64(3110385), np.int64(3110682), np.int64(3110231), np.int64(3111018), np.int64(3110761), np.int64(3130158), np.int64(3110801), np.int64(3110111), np.int64(3110944), np.int64(3130155), np.int64(3130114), np.int64(3110443), np.int64(3110914), np.int64(3110928), np.int64(3110987), np.int64(3110629), np.int64(3130156), np.int64(3110321), np.int64(3110894), np.int64(3110347), np.int64(3130187), np.in